# סימולציית סדרה ובדיקת עבר: גמר קלן 2026

מחברת זו מחברת בין מסווג המפות המבוסס על Elo קנוני לבין סימולציית מונטה קרלו של סדרה מלאה. התוצאה היא הערכה שנוצרה מנקודת המבט שלפני הגמר: דירוגי הקבוצות מוקפאים רגע לפני תחילת הסדרה, ואין שימוש בתוצאות של שלוש המפות ששוחקו בגמר.

## הכנת סביבת העבודה

הזרע האקראי ננעל לצורך שחזור מלא. ששת הפיצ'רים זהים בדיוק למודל הסופי: ארבעה דירוגים מוחלטים ושני הפרשים יחסיים.

In [1]:
from pathlib import Path
from time import perf_counter
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elo_module_dir = PROJECT_ROOT / 'src' / 'features'
if str(elo_module_dir) not in sys.path:
    sys.path.insert(0, str(elo_module_dir))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.splitter import ChronologicalSplitter
from elo import PointInTimeEngine
from src.models.simulator import FrozenEloSnapshot, SeriesSimulator
from src.models.bracket_simulator import (
    TournamentEloState, print_tournament_summary, simulate_tournament,
)

RANDOM_SEED = 42
K_FACTOR = 24.0
LOCKED_N_ESTIMATORS = 76
FEATURE_COLUMNS = [
    'team1_elo_global', 'team2_elo_global',
    'team1_elo_map', 'team2_elo_map',
    'elo_global_diff', 'elo_map_diff',
]

## בניית פיצ'רי Elo בזמן אמת וחלוקה כרונולוגית

מנוע נקודת־הזמן עובר על המפות בסדר כרונולוגי ושומר בכל שורה רק את הדירוג שהיה ידוע לפניה. לאחר מכן מתבצעת החלוקה הנעולה: Train עד סוף 2025, Validation ברבעון הראשון של 2026 ו־Test מהרבעון השני. הכיול יתבצע על Validation בלבד; תוצאות Test אינן נכנסות לאימון או לכיול.

In [2]:
data_path = PROJECT_ROOT / 'data' / 'final_tournament_features.csv'
raw_df = pd.read_csv(data_path, low_memory=False)
point_in_time_df = PointInTimeEngine(k_factor=K_FACTOR).transform(raw_df)
point_in_time_df['elo_global_diff'] = (
    point_in_time_df['team1_elo_global'] - point_in_time_df['team2_elo_global']
)
point_in_time_df['elo_map_diff'] = (
    point_in_time_df['team1_elo_map'] - point_in_time_df['team2_elo_map']
)
splits = ChronologicalSplitter().split(point_in_time_df)
print({name: len(frame) for name, frame in splits.items()})

{'train': 5472, 'val': 633, 'test': 595}


## סימטריה ללא דליפה

Train ו־Validation מסומטרים בנפרד לאחר החלוקה הכרונולוגית. בעותק המראה מוחלפים דירוגי הקבוצות, התווית מתהפכת, ושני פיצ'רי ההפרש מחושבים מחדש ולכן משנים סימן. הפעולה מונעת מהמודל ללמוד הטיה מלאכותית לצד הראשון.

In [3]:
RATING_PAIRS = [
    ('team1_elo_global', 'team2_elo_global'),
    ('team1_elo_map', 'team2_elo_map'),
]

def symmetrize_elo(frame):
    original = frame[FEATURE_COLUMNS + ['team1_win']].copy().reset_index(drop=True)
    mirrored = original.copy()
    for team1_column, team2_column in RATING_PAIRS:
        mirrored[team1_column] = original[team2_column].to_numpy(copy=True)
        mirrored[team2_column] = original[team1_column].to_numpy(copy=True)
    mirrored['team1_win'] = 1 - original['team1_win'].to_numpy()
    symmetric = pd.concat([original, mirrored], ignore_index=True)
    symmetric['elo_global_diff'] = (
        symmetric['team1_elo_global'] - symmetric['team2_elo_global']
    )
    symmetric['elo_map_diff'] = (
        symmetric['team1_elo_map'] - symmetric['team2_elo_map']
    )
    midpoint = len(original)
    for column in ['elo_global_diff', 'elo_map_diff']:
        if not np.allclose(
            symmetric.iloc[:midpoint][column],
            -symmetric.iloc[midpoint:][column],
        ):
            raise AssertionError(f'פיצר ההפרש לא התהפך: {column}')
    return symmetric

train_df = symmetrize_elo(splits['train'])
val_df = symmetrize_elo(splits['val'])
X_train = train_df[FEATURE_COLUMNS]
y_train = train_df['team1_win'].astype(int)
X_val = val_df[FEATURE_COLUMNS]
y_val = val_df['team1_win'].astype(int)
print(f'Train מסומטר: {len(train_df):,}')
print(f'Validation מסומטר: {len(val_df):,}')

Train מסומטר: 10,944
Validation מסומטר: 1,266


## אימון המודל הנעול וכיול איזוטוני

המסווג הבסיסי מאומן רק על Train, עם 76 העצים שננעלו לפני פתיחת Test. לאחר האימון הוא מוקפא, והכיול האיזוטוני לומד על Validation התאמה מונוטונית בין הסתברות חזויה לבין שכיחות ניצחון בפועל. כך הכיול רשאי לשנות את ביטחון התחזית אך לא את סדר הדוגמאות.

בגרסאות החדשות של scikit-learn, `FrozenEstimator` הוא המימוש הרשמי של תרחיש `cv='prefit'`: הוא מבטיח שהמודל הבסיסי לא יאומן מחדש בזמן התאמת המכייל.

In [4]:
canonical_model = XGBClassifier(
    objective='binary:logistic', eval_metric='logloss',
    n_estimators=LOCKED_N_ESTIMATORS, learning_rate=0.03, max_depth=3,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.9,
    reg_lambda=2.0, tree_method='hist', random_state=RANDOM_SEED,
    n_jobs=-1,
)
canonical_model.fit(X_train, y_train, verbose=False)

try:
    from sklearn.frozen import FrozenEstimator
    calibrated_model = CalibratedClassifierCV(
        FrozenEstimator(canonical_model), method='isotonic'
    )
    calibration_api = 'FrozenEstimator'
except ImportError:
    calibrated_model = CalibratedClassifierCV(
        canonical_model, method='isotonic', cv='prefit'
    )
    calibration_api = "cv='prefit'"

calibrated_model.fit(X_val, y_val)
raw_probability = canonical_model.predict_proba(X_val)[:, 1]
calibrated_probability = calibrated_model.predict_proba(X_val)[:, 1]
print(f'מנגנון הקפאת המודל: {calibration_api}')
print(
    f'לפני כיול — לוג־לוס: {log_loss(y_val, raw_probability):.6f}, '
    f'ברייר: {brier_score_loss(y_val, raw_probability):.6f}'
)
print(
    f'אחרי כיול — לוג־לוס: {log_loss(y_val, calibrated_probability):.6f}, '
    f'ברייר: {brier_score_loss(y_val, calibrated_probability):.6f}, '
    f'דיוק: {accuracy_score(y_val, calibrated_probability >= 0.5):.6f}'
)

מנגנון הקפאת המודל: FrozenEstimator
לפני כיול — לוג־לוס: 0.664443, ברייר: 0.236055
אחרי כיול — לוג־לוס: 0.652905, ברייר: 0.231024, דיוק: 0.610585


## שמירת ארטיפקט הטענה

המודל והמכייל נשמרים יחד עם סדר הפיצ'רים. מיד לאחר השמירה הארטיפקט נטען מחדש, כדי שה־backtest יבדוק את אותו מסלול טעינה שבו ישתמש הסימולטור הייצורי.

In [5]:
artifact_path = PROJECT_ROOT / 'artifacts' / 'map_classifier' / 'canonical_elo_isotonic.joblib'
artifact_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {
        'base_model': canonical_model,
        'calibrated_model': calibrated_model,
        'feature_columns': FEATURE_COLUMNS,
        'k_factor': K_FACTOR,
        'n_estimators': LOCKED_N_ESTIMATORS,
    },
    artifact_path,
)
model_artifact = joblib.load(artifact_path)
if model_artifact['feature_columns'] != FEATURE_COLUMNS:
    raise AssertionError('סדר הפיצרים בארטיפקט אינו תואם.')
print(f'הארטיפקט נשמר ונטען: {artifact_path.relative_to(PROJECT_ROOT)}')

הארטיפקט נשמר ונטען: artifacts\map_classifier\canonical_elo_isotonic.joblib


## הקפאת מצב ה־Elo לפני הגמר

נבנה מנוע חדש שמקבל רק מפות שהתרחשו לפני חותמת הזמן של גמר קלן. המילונים בסופו הם תמונת המידע שהייתה זמינה לפני המפה הראשונה. תוצאות הגמר עצמו אינן נגישות למנוע זה. המפתחות `falcons` ו־`furia` הם מזהי הקבוצה הקנוניים, ולכן שינוי במזהה המשחק הבודד אינו מאפס את ההיסטוריה.

In [6]:
raw_datetimes = pd.to_datetime(raw_df['datetime'], errors='raise')
final_mask = raw_df['match_id'].eq(7302876)
if not final_mask.any():
    raise AssertionError('גמר קלן 2026 לא נמצא בקובץ המקור.')
final_time = raw_datetimes.loc[final_mask].iloc[0]
history_before_final = raw_df.loc[raw_datetimes < final_time].copy()
snapshot_engine = PointInTimeEngine(k_factor=K_FACTOR)
snapshot_engine.transform(history_before_final)
snapshot = FrozenEloSnapshot(
    global_ratings=dict(snapshot_engine.global_ratings_),
    map_ratings=dict(snapshot_engine.map_ratings_),
    initial_rating=snapshot_engine.initial_rating,
)
falcons_key = 'falcons'
furia_key = 'furia'
map_sequence = ('Mirage', 'Anubis', 'Inferno', 'Dust2', 'Nuke')
print(f'חותמת זמן של הגמר: {final_time}')
print(f'Elo גלובלי קפוא — Falcons: {snapshot.global_rating(falcons_key):.2f}')
print(f'Elo גלובלי קפוא — FURIA: {snapshot.global_rating(furia_key):.2f}')
print(f'סדר המפות: {map_sequence}')

חותמת זמן של הגמר: 2026-06-21 18:00:00
Elo גלובלי קפוא — Falcons: 1904.59
Elo גלובלי קפוא — FURIA: 1759.69
סדר המפות: ('Mirage', 'Anubis', 'Inferno', 'Dust2', 'Nuke')


## מונטה קרלו עם מומנטום בתוך הסדרה

בכל אחת מ־5,000 הסדרות מתחילים מאותם דירוגים קפואים. בכל מפה המודל והמכייל מפיקים הסתברות לניצחון Falcons, ומתבצעת הגרלה ברנולית. מיד לאחר התוצאה המדומה מחושבת תחזית Elo רגילה ונעשה עדכון עם K=24 לדירוג הגלובלי ולדירוג של אותה מפה. לכן מפה 2 כבר רואה את תוצאת מפה 1, אך אף ריצה אינה חולקת מידע עם ריצה אחרת. הסדרה נעצרת ברגע שקבוצה מגיעה לשלושה ניצחונות.

In [7]:
simulator = SeriesSimulator(
    calibrated_model=model_artifact['calibrated_model'],
    snapshot=snapshot,
    map_sequence=map_sequence,
    k_factor=model_artifact['k_factor'],
    random_state=RANDOM_SEED,
)
result = simulator.simulate(
    team_A=falcons_key, team_B=furia_key, best_of=5, n_iterations=5000
)
print(f'הסתברות ניצחון בסדרה — Falcons: {result.team_a_win_probability:.2%}')
print(f'הסתברות ניצחון בסדרה — FURIA: {result.team_b_win_probability:.2%}')
print('התפלגות תוצאות מדויקות (מנקודת המבט של Falcons):')
for scoreline, probability in result.scoreline_distribution.items():
    print(f'  {scoreline}: {probability:.2%}')
if not np.isclose(sum(result.scoreline_distribution.values()), 1.0):
    raise AssertionError('התפלגות התוצאות אינה מסתכמת ל־100%.')

הסתברות ניצחון בסדרה — Falcons: 70.38%
הסתברות ניצחון בסדרה — FURIA: 29.62%
התפלגות תוצאות מדויקות (מנקודת המבט של Falcons):
  3-0: 27.82%
  3-1: 24.96%
  3-2: 17.60%
  0-3: 4.18%
  1-3: 10.42%
  2-3: 15.02%


## כיצד לקרוא את התוצאה

ההסתברות הכוללת היא סכום ההסתברויות של 3–0, 3–1 ו־3–2 לטובת Falcons. יתר התוצאות הן ניצחונות של FURIA. זהו backtest הסתברותי ולא ניסיון לשחזר בכוח את ה־3–0 שהתרחש בפועל: תוצאה יחידה יכולה להיות אפשרית גם כאשר אינה התוצאה הסבירה ביותר.

# סימולציית טבלת ה־Playoffs המלאה

כעת מרחיבים את אותו מנגנון מסדרה אחת לטורניר הדחה של שמונה קבוצות. בכל אחת מ־10,000 הריצות נוצרת תמונת Elo חדשה וקפואה מלפני רבע הגמר הראשון. מנצחת כל סדרה ממשיכה עם דירוגי ה־Elo שעודכנו במפות שזה עתה שיחקה, ולכן מומנטום מדומה עובר מרבע הגמר לחצי הגמר ומחצי הגמר לגמר. מצב של ריצה אחת לעולם אינו זולג לריצה אחרת.

## מבנה ההצלבות וסדרי המפות

ההצלבות ההיסטוריות הן Aurora–BETBOOM, ‏9z–FURIA, ‏G2–Spirit ו־Falcons–Vitality. מנצחות שני רבעי הגמר הראשונים נפגשות בחצי גמר אחד, ומנצחות שני רבעי הגמר האחרונים בחצי הגמר השני.

לכל צמד שבאמת נפגש בטורניר נשמר סדר ה־veto ההיסטורי של HLTV, כולל מפה מכרעת שלא שוחקה. אם הסימולציה יוצרת צמד שלא התקיים במציאות, סדר המפות נדגם ללא החזרה ממאגר המפות הפעיל: Ancient, Anubis, Dust2, Inferno, Mirage, Nuke ו־Overpass. כך לא מכניסים ידע בדוי על veto שמעולם לא התרחש.

In [8]:
N_TOURNAMENT_ITERATIONS = 10_000
PLAYOFF_START = pd.Timestamp('2026-06-18 16:45:00')
ACTIVE_MAP_POOL = (
    'Ancient', 'Anubis', 'Dust2', 'Inferno', 'Mirage', 'Nuke', 'Overpass'
)
QUARTERFINALS = (
    ('aurora', 'betboom team'),
    ('9z team', 'furia'),
    ('g2', 'spirit'),
    ('falcons', 'vitality'),
)
HISTORICAL_MAP_ORDERS = {
    frozenset(('aurora', 'betboom team')): ('Nuke', 'Anubis', 'Dust2'),
    frozenset(('9z team', 'furia')): ('Dust2', 'Mirage', 'Overpass'),
    frozenset(('g2', 'spirit')): ('Overpass', 'Dust2', 'Mirage'),
    frozenset(('falcons', 'vitality')): ('Anubis', 'Inferno', 'Dust2'),
    frozenset(('aurora', 'furia')): ('Dust2', 'Nuke', 'Inferno'),
    frozenset(('spirit', 'falcons')): ('Anubis', 'Mirage', 'Dust2'),
    frozenset(('furia', 'falcons')): (
        'Mirage', 'Anubis', 'Inferno', 'Dust2', 'Nuke'
    ),
}
PLAYOFF_TEAMS = tuple(team for pairing in QUARTERFINALS for team in pairing)
print(f'מספר קבוצות: {len(PLAYOFF_TEAMS)}')
print(f'מספר סדרי veto היסטוריים: {len(HISTORICAL_MAP_ORDERS)}')

מספר קבוצות: 8
מספר סדרי veto היסטוריים: 7


## תמונת פתיחה לפני ה־Playoffs

בניגוד לבדיקת הגמר הבודד, כאן נקודת הקיפאון מוזזת אל לפני רבע הגמר הראשון. מנוע חדש מעבד רק תוצאות מוקדמות יותר, ולא רואה אף תוצאת Playoffs. מן התמונה המלאה נשמרים רק שמונה המשתתפים ושבע המפות הרלוונטיות, כדי להעתיק מצב קטן ועצמאי ביעילות בכל איטרציה.

In [9]:
history_before_playoffs = raw_df.loc[raw_datetimes < PLAYOFF_START].copy()
playoff_engine = PointInTimeEngine(k_factor=K_FACTOR)
playoff_engine.transform(history_before_playoffs)
playoff_snapshot = FrozenEloSnapshot(
    global_ratings=dict(playoff_engine.global_ratings_),
    map_ratings=dict(playoff_engine.map_ratings_),
    initial_rating=playoff_engine.initial_rating,
)
starting_tournament_state = TournamentEloState.from_snapshot(
    playoff_snapshot, PLAYOFF_TEAMS, ACTIVE_MAP_POOL, k_factor=K_FACTOR
)
starting_ratings = pd.DataFrame({
    'קבוצה': PLAYOFF_TEAMS,
    'Elo גלובלי לפני הפלייאוף': [
        starting_tournament_state.global_rating(team) for team in PLAYOFF_TEAMS
    ],
}).sort_values('Elo גלובלי לפני הפלייאוף', ascending=False)
starting_ratings

,קבוצה,Elo גלובלי לפני הפלייאוף
5,spirit,1946.338492
7,vitality,1910.434902
6,falcons,1819.259677
1,betboom team,1722.360911
0,aurora,1718.547517
4,g2,1714.511066
3,furia,1704.120602
2,9z team,1687.621996


## פונקציה טהורה והעברת מצב

הפרימיטיב `simulate_series_once` אינו משנה את מצב ה־Elo שקיבל. הוא יוצר עותק, מדמה מפות עם עדכון חי לאחר כל מפה, ומחזיר מנצחת, מצב חדש ותוצאה מדויקת. מנוע הטורניר מחליף את המצב הישן במצב המוחזר אחרי כל סדרה. מבנה זה מונע שינוי סמוי של תמונת הפתיחה והופך את זרימת המידע לגלויה וניתנת לבדיקה.

## הרצת 10,000 טורנירים ובדיקות תקינות

בכל ריצה נספרת עצם ההגעה לכל שלב. לולאת מונטה קרלו מחולקת לאצוות ומוצגת באמצעות `tqdm`, כולל קצב ו־ETA. בתוך הסימולציה אין יצירה של אובייקטי Pandas: המודל מקבל מטריצות NumPy ישירות דרך ה־Booster, מצב ה־Elo נשמר במילוני Python, ו־Pandas נכנס לפעולה רק לאחר סיום הריצה כדי לעצב את הטבלה. בסיום נבדק שסכום הסתברויות האליפות הוא 100%, ושעבור כל קבוצה ההסתברויות אינן יכולות לעלות ככל שמתקדמים: רבע גמר ≥ חצי גמר ≥ גמר ≥ אליפות. סטיית התקן של אומדן האליפות מחושבת לפי נוסחת ברנולי, `sqrt(p(1-p)/N)`.

In [10]:
simulation_started = perf_counter()
tournament_results = simulate_tournament(
    quarterfinals=QUARTERFINALS,
    starting_elo_state=starting_tournament_state,
    calibrated_model=model_artifact['calibrated_model'],
    active_map_pool=ACTIVE_MAP_POOL,
    historical_map_orders=HISTORICAL_MAP_ORDERS,
    n_iterations=N_TOURNAMENT_ITERATIONS,
    random_state=RANDOM_SEED,
    batch_size=512,
    show_progress=True,
)
simulation_seconds = perf_counter() - simulation_started
print(f'זמן ריצה מדויק ל־10,000 איטרציות: {simulation_seconds:.3f} שניות')
tournament_summary = print_tournament_summary(
    tournament_results, N_TOURNAMENT_ITERATIONS
)
tournament_summary

Tournament Monte Carlo:   0%|          | 0/10000 [00:00<?, ?iter/s]

זמן ריצה מדויק ל־10,000 איטרציות: 10.459 שניות
    Team  P(QF) P(SF) P(Final) P(Champion) SE(Champion)
  Spirit 100.0% 78.5%    47.4%       37.1%         0.5%
Vitality 100.0% 59.9%    28.8%       22.3%         0.4%
 Falcons 100.0% 40.1%    17.0%       11.1%         0.3%
 BETBOOM 100.0% 57.6%    32.6%        8.7%         0.3%
   FURIA 100.0% 58.6%    26.7%        7.2%         0.3%
  Aurora 100.0% 42.4%    23.9%        6.3%         0.2%
      G2 100.0% 21.5%     6.8%        3.7%         0.2%
      9z 100.0% 41.4%    16.7%        3.5%         0.2%


,Team,P(QF),P(SF),P(Final),P(Champion),SE(Champion)
0,Spirit,100.0%,78.5%,47.4%,37.1%,0.5%
1,Vitality,100.0%,59.9%,28.8%,22.3%,0.4%
2,Falcons,100.0%,40.1%,17.0%,11.1%,0.3%
3,BETBOOM,100.0%,57.6%,32.6%,8.7%,0.3%
4,FURIA,100.0%,58.6%,26.7%,7.2%,0.3%
5,Aurora,100.0%,42.4%,23.9%,6.3%,0.2%
6,G2,100.0%,21.5%,6.8%,3.7%,0.2%
7,9z,100.0%,41.4%,16.7%,3.5%,0.2%


## פרשנות זהירה

הטבלה היא תחזית מנקודת הזמן שלפני שלב ההדחה, ולא דירוג בדיעבד. `P(QF)` שווה 100% מפני שכל שמונה הקבוצות כבר נכנסו לרבע הגמר. שאר העמודות מודדות את שיעור הריצות שבהן הקבוצה הגיעה לשלב המתאים. `SE(Champion)` מתאר את אי־הוודאות של מונטה קרלו בלבד; הוא אינו כולל אי־ודאות בבחירת המודל או בדאטה.

# ביקורת זהויות קנוניות

בדיקת המקור העלתה שהמפתחות הקנוניים הם `9z team` ו־`betboom team`, ושכך הם מופיעים בעקביות ב־Train, ב־Validation וב־Test. קבוצות Academy נשמרות במפתחות נפרדים ואינן מתמזגות עם הארגון הראשי. בדיקת Point‑in‑Time נוספת הוכיחה שאין אף שורת Test של שמונה משתתפות ה־Playoffs שבה ה־Global Elo התאפס ל־1500. לכן התקלה הייתה מקומית למילון החיפוש הידני של הסימולטור, ואין לפתוח מחדש את מדד ה־Test הנעול.

# שלב 2: מבחן יציבות עם 100,000 טורנירים

אותו טורניר מורץ פעמיים, פעם עם seed ‏42 ופעם עם seed ‏99. כל ריצה כוללת 100,000 מסלולים ומוצגת בסרגל `tqdm` עצמאי. לאחר מכן מחושב לכל קבוצה ההפרש המוחלט בהסתברות האליפות בין הזרעים. תנאי הקבלה הוא שההפרש המרבי לא יעלה על 0.5 נקודת אחוז.

In [11]:
STABILITY_ITERATIONS = 100_000

def run_stability_seed(seed):
    started = perf_counter()
    results = simulate_tournament(
        quarterfinals=QUARTERFINALS,
        starting_elo_state=starting_tournament_state,
        calibrated_model=model_artifact['calibrated_model'],
        active_map_pool=ACTIVE_MAP_POOL,
        historical_map_orders=HISTORICAL_MAP_ORDERS,
        n_iterations=STABILITY_ITERATIONS,
        random_state=seed,
        batch_size=2_048,
        show_progress=True,
    )
    elapsed = perf_counter() - started
    print(f'זמן ריצה עבור seed={seed}: {elapsed:.3f} שניות')
    return results, elapsed

results_seed_42, seconds_seed_42 = run_stability_seed(42)
results_seed_99, seconds_seed_99 = run_stability_seed(99)

champion_stability = pd.DataFrame([
    {
        'Team': team,
        'P(Champion), seed=42': results_seed_42[team]['Champion'] / STABILITY_ITERATIONS,
        'P(Champion), seed=99': results_seed_99[team]['Champion'] / STABILITY_ITERATIONS,
    }
    for team in PLAYOFF_TEAMS
])
champion_stability['Absolute Difference'] = (
    champion_stability['P(Champion), seed=42']
    - champion_stability['P(Champion), seed=99']
).abs()
max_champion_difference = champion_stability['Absolute Difference'].max()
assert max_champion_difference <= 0.005, (
    f'הפרש היציבות המרבי חרג מ־0.5%: {max_champion_difference:.3%}'
)
print(f'הפרש מוחלט מרבי ב־P(Champion): {max_champion_difference:.3%}')
champion_stability = champion_stability.sort_values(
    'P(Champion), seed=42', ascending=False
).reset_index(drop=True)
for column in ['P(Champion), seed=42', 'P(Champion), seed=99', 'Absolute Difference']:
    champion_stability[column] = champion_stability[column].map(lambda value: f'{value:.3%}')
champion_stability

Tournament Monte Carlo:   0%|          | 0/100000 [00:00<?, ?iter/s]

זמן ריצה עבור seed=42: 93.398 שניות


Tournament Monte Carlo:   0%|          | 0/100000 [00:00<?, ?iter/s]

זמן ריצה עבור seed=99: 92.639 שניות
הפרש מוחלט מרבי ב־P(Champion): 0.306%


,Team,"P(Champion), seed=42","P(Champion), seed=99",Absolute Difference
0,spirit,37.420%,37.114%,0.306%
1,vitality,22.188%,22.374%,0.186%
2,falcons,11.268%,11.205%,0.063%
3,betboom team,8.320%,8.183%,0.137%
4,furia,7.080%,7.221%,0.141%
5,aurora,6.253%,6.313%,0.060%
6,g2,4.057%,3.997%,0.060%
7,9z team,3.414%,3.593%,0.179%


## טבלת הסיכום ברזולוציה הגבוהה

הטבלה הבאה משתמשת בריצת seed ‏42 כטבלת הייחוס המלאה. ריצת seed ‏99 משמשת כבקרת יציבות בלתי תלויה, ולא לבחירה בדיעבד של תוצאה נוחה יותר.

In [12]:
summary_100k_seed_42 = print_tournament_summary(
    results_seed_42, STABILITY_ITERATIONS
)
summary_100k_seed_42

    Team  P(QF) P(SF) P(Final) P(Champion) SE(Champion)
  Spirit 100.0% 78.4%    47.4%       37.4%         0.2%
Vitality 100.0% 60.1%    28.7%       22.2%         0.1%
 Falcons 100.0% 39.9%    17.0%       11.3%         0.1%
 BETBOOM 100.0% 56.7%    32.5%        8.3%         0.1%
   FURIA 100.0% 59.1%    26.8%        7.1%         0.1%
  Aurora 100.0% 43.3%    24.2%        6.3%         0.1%
      G2 100.0% 21.6%     6.9%        4.1%         0.1%
      9z 100.0% 40.9%    16.6%        3.4%         0.1%


,Team,P(QF),P(SF),P(Final),P(Champion),SE(Champion)
0,Spirit,100.0%,78.4%,47.4%,37.4%,0.2%
1,Vitality,100.0%,60.1%,28.7%,22.2%,0.1%
2,Falcons,100.0%,39.9%,17.0%,11.3%,0.1%
3,BETBOOM,100.0%,56.7%,32.5%,8.3%,0.1%
4,FURIA,100.0%,59.1%,26.8%,7.1%,0.1%
5,Aurora,100.0%,43.3%,24.2%,6.3%,0.1%
6,G2,100.0%,21.6%,6.9%,4.1%,0.1%
7,9z,100.0%,40.9%,16.6%,3.4%,0.1%
